In [2]:
%pip install scikit-learn xgboost

  Obtaining dependency information for scikit-learn from https://files.pythonhosted.org/packages/97/74/b7a304feb2b49df9fafa9382d4d09061a96ee9a9449a7cbea7988dda0828/scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for xgboost from https://files.pythonhosted.org/packages/79/98/679de17c2caa4fd3b0b4386ecf7377301702cb0afb22930a07c142fcb1d8/xgboost-3.2.0-py3-none-manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for scipy>=1.10.0 from https://files.pythonhosted.org/packages/01/8e/1e35281b8ab6d5d72ebe9911edcdffa3f36b04ed9d51dec6dd140396e220/scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 7.3 MB/s eta 0:00:00
  Obtaining dependency information for joblib>=1.3.0 from https://files.pythonhosted.org/packages/7b/91/984aca2ec129e2757d1e4e3c81c3fcda9d0f85b74670a094cc443d9ee949/joblib-1.5.3-py3-none-any.whl.metadata


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Cargar el dataset enriquecido
print("Cargando matriz de características...")
df = pd.read_csv('../data/processed/features_finales.csv')

# 2. Aislar las variables predictivas (X) y la variable objetivo (y)
features = ['elo_home', 'elo_away', 'neutral_numeric']
X = df[features]
y = df['target_numeric']

# 3. Train-Test Split Cronológico (Sin mezclar el tiempo)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Set de Entrenamiento: {len(X_train)} partidos antiguos.")
print(f"Set de Validación: {len(X_test)} partidos recientes.")

# 4. Inicializar y Entrenar el Modelo XGBoost
print("\nEntrenando algoritmo XGBoost...")
modelo_xgb = XGBClassifier(
    n_estimators=100,      # Número de árboles
    learning_rate=0.1,     # Tasa de aprendizaje
    max_depth=4,           # Profundidad para evitar sobreajuste
    random_state=42
)

modelo_xgb.fit(X_train, y_train)

# 5. Predicción y Evaluación
y_pred = modelo_xgb.predict(X_test)

print("\n--- Rendimiento del Modelo ---")
print(f"Precisión Global (Accuracy): {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Reporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=['Pierde Local (0)', 'Empate (1)', 'Gana Local (2)']))

Cargando matriz de características...
Set de Entrenamiento: 20082 partidos antiguos.
Set de Validación: 5021 partidos recientes.

Entrenando algoritmo XGBoost...

--- Rendimiento del Modelo ---
Precisión Global (Accuracy): 59.75%

Reporte de Clasificación:
                  precision    recall  f1-score   support

Pierde Local (0)       0.55      0.64      0.59      1492
      Empate (1)       0.45      0.01      0.02      1156
  Gana Local (2)       0.62      0.86      0.72      2373

        accuracy                           0.60      5021
       macro avg       0.54      0.50      0.44      5021
    weighted avg       0.56      0.60      0.52      5021



In [ ]:
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML

# 1. Reconstruir la memoria rápida: El Elo más reciente de cada equipo
equipos_unicos = sorted(list(set(df['home_team']).union(set(df['away_team']))))
ultimo_elo = {}

for equipo in equipos_unicos:
    # Extraer el último partido registrado de cada selección
    filtro = df[(df['home_team'] == equipo) | (df['away_team'] == equipo)]
    if not filtro.empty:
        ultimo_partido = filtro.iloc[-1]
        if ultimo_partido['home_team'] == equipo:
            ultimo_elo[equipo] = ultimo_partido['elo_home']
        else:
            ultimo_elo[equipo] = ultimo_partido['elo_away']

# 2. El Motor de Simulación Cuantitativa
def simular_partido(equipo_1, equipo_2):
    if equipo_1 == equipo_2:
        return print("Error: Un equipo no puede jugar contra sí mismo.")
    
    elo_1 = ultimo_elo.get(equipo_1, 1500)
    elo_2 = ultimo_elo.get(equipo_2, 1500)
    
    # En un Mundial, la ventaja de localía se anula (cancha neutral = 1)
    X_nuevo = pd.DataFrame({
        'elo_home': [elo_1],
        'elo_away': [elo_2],
        'neutral_numeric': [1]
    })
    
    # El algoritmo piensa...
    prediccion = modelo_xgb.predict(X_nuevo)[0]
    probabilidades = modelo_xgb.predict_proba(X_nuevo)[0]
    
    # Formatear el reporte de salida
    print(f"\n🏆 SIMULADOR MUNDIAL 2026 🏆")
    print("-" * 40)
    print(f"[{equipo_1}] vs [{equipo_2}]")
    print(f"Fuerza Matemática Actual: {elo_1:.0f} vs {elo_2:.0f}")
    print("-" * 40)
    
    if prediccion == 2:
        print(f"🔮 PREDICCIÓN DEL MODELO: Victoria de {equipo_1}")
    elif prediccion == 0:
        print(f"🔮 PREDICCIÓN DEL MODELO: Victoria de {equipo_2}")
    else:
        print(f"🔮 PREDICCIÓN DEL MODELO: Empate (Se define por penales)")
        
    print(f"\n📊 Desglose de Probabilidades:")
    print(f"Gana {equipo_1}: {probabilidades[2]*100:.1f}%")
    print(f"Gana {equipo_2}: {probabilidades[0]*100:.1f}%")
    print(f"Empate:       {probabilidades[1]*100:.1f}%")

# 3. Construcción de la Interfaz Gráfica
dropdown_1 = widgets.Dropdown(options=equipos_unicos, value='Argentina', description='Local:')
dropdown_2 = widgets.Dropdown(options=equipos_unicos, value='France', description='Visitante:')
boton = widgets.Button(description='Simular Choque', button_style='success')
salida = widgets.Output()

def al_hacer_click(b):
    with salida:
        salida.clear_output()
        simular_partido(dropdown_1.value, dropdown_2.value)

boton.on_click(al_hacer_click)

# Mostrar el widget en pantalla
display(HTML("<h3>⚽ El Oráculo del Mundial</h3>"))
display(widgets.HBox([dropdown_1, dropdown_2]))
display(boton, salida)

Button(button_style='success', description='Simular Choque', style=ButtonStyle())

Output()

In [6]:
import joblib
import os

os.makedirs('../models', exist_ok=True)

# 1. Guardamos el modelo XGBoost entrenado
joblib.dump(modelo_xgb, '../models/xgboost_worldcup.pkl')

# 2. Guardamos el diccionario con el último Elo de cada equipo
joblib.dump(ultimo_elo, '../models/ultimo_elo.pkl')

print("¡Cerebro exportado! Listo para ser consumido por Streamlit.")

¡Cerebro exportado! Listo para ser consumido por Streamlit.


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import joblib
import os

print("1. Cargando datos enriquecidos...")
df = pd.read_csv('../data/processed/features_finales.csv')

# 2. Reestructurar las variables objetivo (Target)
X = df[['elo_home', 'elo_away', 'neutral_numeric']]
y_home = df['home_score']  # Lo que intentará predecir el Modelo A
y_away = df['away_score']  # Lo que intentará predecir el Modelo B

# 3. Train-Test Split Cronológico
X_train, X_test, y_home_train, y_home_test = train_test_split(X, y_home, test_size=0.2, shuffle=False)
_, _, y_away_train, y_away_test = train_test_split(X, y_away, test_size=0.2, shuffle=False)

print("2. Entrenando Motor de Regresión A (Goles Local)...")
regresor_local = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
regresor_local.fit(X_train, y_home_train)

print("3. Entrenando Motor de Regresión B (Goles Visitante)...")
regresor_visitante = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
regresor_visitante.fit(X_train, y_away_train)

# 4. Evaluación de Error Absoluto Medio (MAE)
pred_home = regresor_local.predict(X_test)
pred_away = regresor_visitante.predict(X_test)

mae_home = mean_absolute_error(y_home_test, pred_home)
mae_away = mean_absolute_error(y_away_test, pred_away)

print("\n--- Rendimiento de los Regresores ---")
print(f"Margen de error Goles Local: +/- {mae_home:.2f} goles por partido")
print(f"Margen de error Goles Visitante: +/- {mae_away:.2f} goles por partido")

# 5. Exportar el nuevo cerebro a disco
os.makedirs('../models', exist_ok=True)
joblib.dump(regresor_local, '../models/xgb_reg_home.pkl')
joblib.dump(regresor_visitante, '../models/xgb_reg_away.pkl')
print("\n¡Nuevos modelos de regresión guardados con éxito!")

1. Cargando datos enriquecidos...
2. Entrenando Motor de Regresión A (Goles Local)...
3. Entrenando Motor de Regresión B (Goles Visitante)...

--- Rendimiento de los Regresores ---
Margen de error Goles Local: +/- 1.05 goles por partido
Margen de error Goles Visitante: +/- 0.86 goles por partido

¡Nuevos modelos de regresión guardados con éxito!
